# Assignment 1

In this project I take data from Wikipedia and save it into database.

Then I use this data for website.

In [77]:
# import libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import re
import time

## Get main page

In [78]:
url = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

print("page loaded")

page loaded


## Find table

In [79]:
table = soup.find("table", class_="wikitable")
rows = table.find_all("tr")

print("rows:", len(rows))

rows: 51


## Find columns

In [80]:
headers_list = [h.get_text(strip=True) for h in rows[0].find_all(["th","td"])]

def find_col(name):
    for i, h in enumerate(headers_list):
        if name.lower() in h.lower():
            return i
    return None

title_i = find_col("title")
gross_i = find_col("gross")
year_i = find_col("year")

print(title_i, gross_i, year_i)

2 3 4


## Helper functions

In [81]:
def clean_country_text(text):
    if not text:
        return "Unknown"

    # remove references like [1], [2], [4]
    text = re.sub(r"\[\s*\d+\s*\]", "", text)

    # fix spaces
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()

    # split countries if they are written together
    parts = re.split(r"\s{2,}|,|\n", text)

    clean_parts = []
    for p in parts:
        p = p.strip()
        if p and p not in clean_parts:
            clean_parts.append(p)

    if clean_parts:
        return ", ".join(clean_parts)

    return text if text else "Unknown"


# clean text
def clean(t):
    if not t:
        return None
    t = re.sub(r"\[[^\]]*\]", "", t)
    t = t.replace("\xa0"," ")
    return t.strip()
def fetch_page(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=10)
    return response.text
# get year
def get_year(t):
    m = re.search(r"\d{4}", t)
    return int(m.group()) if m else None

# get money
def get_money(t):
    m = re.search(r"\$[\d,]+", t)
    return m.group() if m else t

# get link
def get_link(cell):
    a = cell.find("a")
    if a and a.get("href"):
        return "https://en.wikipedia.org" + a["href"]
    return None
def extract_film_details(film_url):
    try:
        soup = BeautifulSoup(fetch_page(film_url), "html.parser")

        infobox = soup.find("table", class_=lambda c: c and "infobox" in c)
        if not infobox:
            return "Unknown", "Unknown"

        director = "Unknown"
        country = "Unknown"

        for row in infobox.find_all("tr"):
            th = row.find("th")
            td = row.find("td")

            if not th or not td:
                continue

            text = th.get_text(" ", strip=True).lower()

            # director
            if "directed by" in text:
                names = []
                for a in td.find_all("a"):
                    txt = a.get_text(" ", strip=True)
                    href = a.get("href", "")

                    if not txt:
                        continue
                    if txt.startswith("[") and txt.endswith("]"):
                        continue
                    if href.startswith("#"):
                        continue
                    if ":" in href:
                        continue

                    if txt not in names:
                        names.append(txt)

                if names:
                    director = ", ".join(names)
                else:
                    director = td.get_text(" ", strip=True)

            # country
            elif text in ["country", "countries", "country of origin"]:
                raw_country = td.get_text(" ", strip=True)
                country = clean_country_text(raw_country)

        return director, country

    except:
        return "Unknown", "Unknown"

## Get data from film page

In [82]:
def get_info(soup, label):
    box = soup.find("table", class_=lambda x: x and "infobox" in x)
    if not box:
        return "Unknown"

    for r in box.find_all("tr"):
        th = r.find("th")
        td = r.find("td")

        if th and td:
            if label.lower() in th.get_text().lower():
                return clean(td.get_text(" ", strip=True))

    return "Unknown"

## Parse films

In [83]:
films = []

for r in rows[1:]:
    cols = r.find_all(["td","th"])

    if len(cols) < 5:
        continue

    title_cell = cols[title_i]

    title = clean(title_cell.get_text())
    year = get_year(cols[year_i].get_text())
    money = get_money(cols[gross_i].get_text())

    link = get_link(title_cell)

    director = "Unknown"
    country = "Unknown"

    if link:
        try:
            director, country = extract_film_details(link)
            print("ok:", title)

        except:
            print("fail:", title)

        time.sleep(0.4)

    films.append({
        "title": title,
        "release_year": year,
        "director": director,
        "box_office": money,
        "country": country
    })


print("done:", len(films))

ok: Avatar
ok: Avengers: Endgame
ok: Avatar: The Way of Water
ok: Titanic
ok: Ne Zha 2
ok: Star Wars: The Force Awakens
ok: Avengers: Infinity War
ok: Spider-Man: No Way Home
ok: Zootopia 2 †
ok: Inside Out 2
ok: Jurassic World
ok: The Lion King
ok: The Avengers
ok: Furious 7
ok: Top Gun: Maverick
ok: Avatar: Fire and Ash †
ok: Frozen 2
ok: Barbie
ok: Avengers: Age of Ultron
ok: The Super Mario Bros. Movie
ok: Black Panther
ok: Harry Potter and the Deathly Hallows – Part 2
ok: Deadpool & Wolverine
ok: Star Wars: The Last Jedi
ok: Jurassic World: Fallen Kingdom
ok: Frozen
ok: Beauty and the Beast
ok: Incredibles 2
ok: The Fate of the Furious
ok: Iron Man 3
ok: Minions
ok: Captain America: Civil War
ok: Aquaman
ok: The Lord of the Rings: The Return of the King
ok: Spider-Man: Far From Home
ok: Captain Marvel
ok: Transformers: Dark of the Moon
ok: Skyfall
ok: Transformers: Age of Extinction
ok: The Dark Knight Rises
ok: Joker
ok: Star Wars: The Rise of Skywalker
ok: Toy Story 4
ok: Toy St

## Create dataframe

In [84]:
df = pd.DataFrame(films)

df.head()

,title,release_year,director,box_office,country
0,Avatar,2009,James Cameron,"$2,923,710,708",United States United Kingdom
1,Avengers: Endgame,2019,Anthony Russo Joe Russo,"$2,797,501,328",United States
2,Avatar: The Way of Water,2022,James Cameron,"$2,334,484,620",United States
3,Titanic,1997,James Cameron,"$2,257,906,828",United States
4,Ne Zha 2,2025,Jiaozi,"$2,215,690,000",China


## Save to database

In [85]:
conn = sqlite3.connect("films.db")

df.to_sql("films", conn, if_exists="replace", index=False)

conn.close()

## Save to json

In [86]:
df.to_json("films.json", orient="records", indent=4)

print("saved")

saved
